# Deep Learning Training Notebook (Colab)

**Before you run anything:** `Runtime > Change runtime type > T4 GPU`. Training on
Colab's CPU is roughly 20-50x slower and will waste your afternoon.

Run the cells top to bottom. You only need to edit two of them, marked **TODO 1**
(your data) and **TODO 2** (your model). Everything else works as-is.

## 1. Check the environment

In [ ]:
import torch, sys

print("python  ", sys.version.split()[0])
print("torch   ", torch.__version__)
print("cuda    ", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu     ", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"vram     {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
else:
    print("\n!! No GPU. Runtime > Change runtime type > T4 GPU, then rerun.")

## 2. Persist your checkpoints

Colab deletes everything in `/content` when the session ends, and free sessions get
disconnected without warning. If a 3-hour run dies at hour 2 and the checkpoint was
in `/content`, that work is gone.

Mounting Drive means checkpoints survive, and the training cell below can resume
from where it stopped. Set `USE_DRIVE = False` if you're doing a quick experiment
you don't mind losing.

In [ ]:
from pathlib import Path

USE_DRIVE = True
RUN_NAME  = "exp001"          # bump this for each experiment - do not overwrite runs

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/dl_course") / RUN_NAME
else:
    OUT_DIR = Path("/content/runs") / RUN_NAME

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("saving to:", OUT_DIR)

## 3. Config and reproducibility

In [ ]:
import random, json, time
import numpy as np
import torch
import torch.nn as nn
from dataclasses import dataclass, asdict
from torch.utils.data import DataLoader

@dataclass
class Config:
    epochs: int        = 20
    batch_size: int    = 128
    lr: float          = 1e-3
    weight_decay: float= 1e-4
    seed: int          = 42
    patience: int      = 5     # stop after this many epochs with no val improvement
    num_workers: int   = 2     # Colab gives you 2 CPU cores; more is not faster

cfg = Config()

def set_seed(seed):
    """Call before building data, model, or optimizer - order matters."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Record the config next to the results, or you will not remember what exp001 was.
(OUT_DIR / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
print(device, asdict(cfg))

## 4. TODO 1 - Your data

This runs on FashionMNIST as a working example. Replace it with your dataset.

Two things to get right:

- **Normalisation statistics come from the training set only.** Computing mean/std
  over train+test leaks test information into training and inflates your scores.
- **Augmentation goes on train, never on val or test.** Your validation number
  should measure the model, not a random crop.

In [ ]:
from torchvision import datasets, transforms

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),                 # augmentation: TRAIN ONLY
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),        # FashionMNIST train stats
])

eval_tf = transforms.Compose([                         # no augmentation here
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_ds = datasets.FashionMNIST("/content/data", train=True,  download=True, transform=train_tf)
val_ds   = datasets.FashionMNIST("/content/data", train=False, download=True, transform=eval_tf)

N_CLASSES   = 10
INPUT_SHAPE = (1, 28, 28)
CLASS_NAMES = train_ds.classes

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)

xb, yb = next(iter(train_loader))
print(f"train {len(train_ds)}  val {len(val_ds)}  batch {tuple(xb.shape)}  labels {tuple(yb.shape)}")
print("range", f"{xb.min():.2f} to {xb.max():.2f}")   # sanity check: should be roughly -1 to 3

## 5. TODO 2 - Your model

Run this baseline first and write down its accuracy. Without that number you cannot
claim any later change helped, and "I improved the architecture" with nothing to
compare against is worth very little in a report.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, in_ch, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),    nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),    nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),    nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(64, n_classes),
        )
    def forward(self, x):
        return self.head(self.features(x))

set_seed(cfg.seed)                       # reseed so the model init is reproducible
model = SmallCNN(INPUT_SHAPE[0], N_CLASSES).to(device)

print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable params")
print("output shape", model(xb[:2].to(device)).shape)    # must be (2, N_CLASSES)

## 6. Training and evaluation loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """One pass. Passing an optimizer = train mode; omitting it = eval mode."""
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = model(x)
            loss   = criterion(logits, y)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                # Uncomment if loss becomes nan or spikes (common with RNNs):
                # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * y.size(0)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)

    return total_loss / total, correct / total

## 7. Train

Saves after every epoch, so a disconnect costs you one epoch rather than the run.
Rerun this cell after reconnecting and it picks up where it left off.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                       factor=0.5, patience=2)

CKPT    = OUT_DIR / "last.pt"
BEST    = OUT_DIR / "best.pt"
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
start_epoch, best_val, bad = 1, float("inf"), 0

if CKPT.exists():                                   # resume after a disconnect
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optim"])
    scheduler.load_state_dict(ck["sched"]); history = ck["history"]
    start_epoch, best_val = ck["epoch"] + 1, ck["best_val"]
    print(f"resumed from epoch {ck['epoch']}")

for epoch in range(start_epoch, cfg.epochs + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader,   criterion)
    scheduler.step(va_loss)

    for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc]):
        history[k].append(v)

    tag = ""
    if va_loss < best_val - 1e-4:
        best_val, bad, tag = va_loss, 0, "  <- best"
        torch.save({"epoch": epoch, "model": model.state_dict(),
                    "val_loss": va_loss, "val_acc": va_acc}, BEST)
    else:
        bad += 1

    torch.save({"epoch": epoch, "model": model.state_dict(),
                "optim": optimizer.state_dict(), "sched": scheduler.state_dict(),
                "history": history, "best_val": best_val}, CKPT)

    print(f"epoch {epoch:3d}/{cfg.epochs}  "
          f"train {tr_loss:.4f}/{tr_acc:.3f}  val {va_loss:.4f}/{va_acc:.3f}  "
          f"lr {optimizer.param_groups[0]['lr']:.2e}  {time.time()-t0:.0f}s{tag}")

    if bad >= cfg.patience:
        print(f"early stop: no improvement for {cfg.patience} epochs")
        break

(OUT_DIR / "history.json").write_text(json.dumps(history, indent=2))
print(f"\nbest val loss {min(history['val_loss']):.4f} | best val acc {max(history['val_acc']):.3f}")

## 8. Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, m in zip(axes, ["loss", "acc"]):
    ax.plot(history[f"train_{m}"], label="train")
    ax.plot(history[f"val_{m}"],   label="val")
    ax.set_xlabel("epoch"); ax.set_ylabel(m); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "curves.png", dpi=150)     # put this figure in your report
plt.show()

**Reading these curves:**

| What you see | What it means | What to do |
|---|---|---|
| Val loss rises while train loss falls | Overfitting | More augmentation, more dropout, weight decay, or stop earlier |
| Both plateau high | Underfitting | Bigger model, higher LR, train longer |
| Loss spikes or goes nan | LR too high, or exploding gradients | Lower LR 10x, enable gradient clipping |
| Val consistently better than train | Normal with dropout/augmentation | Nothing - this is expected |

## 9. Per-class results

In [ ]:
model.load_state_dict(torch.load(BEST, map_location=device)["model"])   # best, not last
model.eval()

preds, targets = [], []
with torch.no_grad():
    for x, y in val_loader:
        preds.append(model(x.to(device)).argmax(1).cpu())
        targets.append(y)
preds, targets = torch.cat(preds), torch.cat(targets)

print(f"overall accuracy {(preds == targets).float().mean():.4f}\n")
for i, name in enumerate(CLASS_NAMES):
    mask = targets == i
    print(f"{name:<15} {(preds[mask] == i).float().mean():.3f}   (n={mask.sum().item()})")

In [ ]:
cm = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long)
for t, p in zip(targets, preds):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=90)
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix")
fig.tight_layout(); fig.savefig(OUT_DIR / "confusion.png", dpi=150); plt.show()

## Colab notes

**Sessions die.** Free tier disconnects after roughly 90 minutes idle and caps at
about 12 hours total. The checkpointing above means you rerun the training cell and
continue rather than restart. Keep the tab open and interact occasionally.

**Runtime restarts wipe variables, not Drive.** After a restart, rerun cells 1-6,
then the training cell resumes from `last.pt`.

**GPU quota is not unlimited.** Free tier throttles you if you use it heavily. Debug
with a small subset on CPU, then switch to GPU for the real run:
`torch.utils.data.Subset(train_ds, range(500))`.

**Change one thing at a time.** Bump `RUN_NAME` for each experiment and log what you
changed. Two changes at once means you learn nothing from the result, and your report
needs the comparison table.

**If it errors:** shape mismatches are the usual cause. Print `x.shape` before the
failing layer. `Expected 4D got 2D` means a flatten happened too early; a size
mismatch in the final linear layer means your `AdaptiveAvgPool2d` output does not
match its `in_features`.